In [1]:
try:
    from google.colab import drive  # Colab only
except Exception:
    drive = None
import os as _os
if drive is not None and not _os.path.ismount("/content/drive"):
    drive.mount("/content/drive")  # [sanitized] optional; data paths are relative to DrugReview_ROOT

import sys
from pathlib import Path
import os
import pandas as pd

DrugReview_ROOT = Path(".")
sys.path.append(str(DrugReview_ROOT))



RAW_DATA_FILE = "Text_Metadata/drugsCom_mood_anxiety_with_ai_labels_mini.csv"
DATA_PATH = os.path.join(DrugReview_ROOT, "data", RAW_DATA_FILE)  # or os.path.join(PROJECT_PATH, RAW_DATA_FILE)
df = pd.read_csv(DATA_PATH)

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix, classification_report, cohen_kappa_score, accuracy_score
)

# ---------------------------
# Config
# ---------------------------
HUMAN_COL = "sentiment_5"         # human 5-class
AI_HARD_COL = "ai_sentiment_5"    # GPT hard 5-class
AI_RATING_COL = "ai_rating_10"    # GPT 1..10

PROB_COLS = [
    "ai_prob_very_negative",
    "ai_prob_negative",
    "ai_prob_neutral",
    "ai_prob_positive",
    "ai_prob_very_positive",
]

ORDER5 = ["very_negative", "negative", "neutral", "positive", "very_positive"]
LABEL2IDX = {lab:i for i,lab in enumerate(ORDER5)}
IDX2LABEL = {i:lab for lab,i in LABEL2IDX.items()}

# ---------------------------
# Helpers
# ---------------------------
def norm5(x):
    if pd.isna(x):
        return np.nan
    return (str(x).strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_"))

def rating10_to_sent5(r):
    if pd.isna(r):
        return np.nan
    r = int(r)
    if r <= 2:  return "very_negative"
    if r <= 4:  return "negative"
    if r <= 6:  return "neutral"
    if r <= 8:  return "positive"
    return "very_positive"

def defensive_renorm(p, eps=1e-12):
    """Clip + renormalize rows to sum to 1 (numerical tolerance)."""
    p = np.clip(p, eps, 1.0)
    row_sums = p.sum(axis=1, keepdims=True)
    return p / row_sums

def entropy_norm(p):
    """Normalized Shannon entropy in [0,1] (natural log)."""
    K = p.shape[1]
    H = -np.sum(p * np.log(p), axis=1)
    return H / np.log(K)

def multiclass_brier(p, y_idx):
    """Mean sum_k (p_k - 1[y=k])^2."""
    n, K = p.shape
    y_onehot = np.zeros((n, K), dtype=float)
    y_onehot[np.arange(n), y_idx] = 1.0
    return np.mean(np.sum((p - y_onehot)**2, axis=1))

# ---------------------------
# 1) Normalize labels
# ---------------------------
df[HUMAN_COL] = df[HUMAN_COL].map(norm5)
df[AI_HARD_COL] = df[AI_HARD_COL].map(norm5)

# rating-derived 5-class from GPT rating_10
df["ai_sent5_from_rating10"] = df[AI_RATING_COL].apply(rating10_to_sent5)

# ---------------------------
# 2) Build probability matrix + argmax label
# ---------------------------
missing = [c for c in PROB_COLS if c not in df.columns]
assert len(missing) == 0, f"Missing probability columns: {missing}"

p_raw = df[PROB_COLS].astype(float).to_numpy()
p = defensive_renorm(p_raw)

df["ai_prob_argmax"] = np.argmax(p, axis=1)
df["ai_sent5_from_probs"] = df["ai_prob_argmax"].map(IDX2LABEL)

df["ai_pmax"] = p.max(axis=1)
df["ai_entropy_norm"] = entropy_norm(p)

# ---------------------------
# 3) HARD-LABEL AGREEMENT (human vs different AI hard labels)
# ---------------------------
def hard_agreement(y_true, y_pred, name):
    mask = y_true.notna() & y_pred.notna()
    yt = y_true[mask].astype(str)
    yp = y_pred[mask].astype(str)

    acc = accuracy_score(yt, yp)
    qwk = cohen_kappa_score(yt, yp, labels=ORDER5, weights="quadratic")

    cm = confusion_matrix(yt, yp, labels=ORDER5)
    cm_df = pd.DataFrame(cm, index=[f"T:{c}" for c in ORDER5], columns=[f"P:{c}" for c in ORDER5])

    print(f"\n=== {name} ===")
    print(f"N = {len(yt)}")
    print(f"Exact agreement (accuracy): {acc:.4f}")
    print(f"Quadratic weighted kappa:   {qwk:.4f}")
    print("\nConfusion matrix (counts):")
    display(cm_df)
    print("\nClassification report:")
    print(classification_report(yt, yp, labels=ORDER5, zero_division=0))

# A) Human vs GPT hard label
hard_agreement(df[HUMAN_COL], df[AI_HARD_COL], "Human sentiment_5 vs AI hard ai_sentiment_5")

# B) Human vs GPT rating10-binned label
hard_agreement(df[HUMAN_COL], df["ai_sent5_from_rating10"], "Human sentiment_5 vs AI (ai_rating_10 -> 5-class)")

# C) Human vs Prob-argmax label
hard_agreement(df[HUMAN_COL], df["ai_sent5_from_probs"], "Human sentiment_5 vs AI (prob argmax)")

# D) Internal consistency: GPT hard vs prob-argmax
hard_agreement(df[AI_HARD_COL], df["ai_sent5_from_probs"], "AI hard ai_sentiment_5 vs AI prob-argmax")

# ---------------------------
# 4) SOFT-LABEL METRICS (human vs probability vector)
# ---------------------------
mask_soft = df[HUMAN_COL].notna()
y_true = df.loc[mask_soft, HUMAN_COL].map(LABEL2IDX).astype(int).to_numpy()
p_soft = p[mask_soft.to_numpy()]

# Prob assigned to the human class
p_true = p_soft[np.arange(len(y_true)), y_true]
mean_p_true = float(np.mean(p_true))

# NLL of the human class (cross-entropy on observed label)
nll = float(np.mean(-np.log(np.clip(p_true, 1e-12, 1.0))))

# Multiclass Brier
brier = float(multiclass_brier(p_soft, y_true))

# Ordinal expected class index + error
idx = np.arange(5)
exp_idx = p_soft @ idx
mae = float(np.mean(np.abs(exp_idx - y_true)))
rmse = float(np.sqrt(np.mean((exp_idx - y_true)**2)))

# Expected absolute ordinal error (W1 to one-hot along ordinal axis)
# (equivalent here to E[|K - y|] under p)
w1 = float(np.mean(np.sum(p_soft * np.abs(idx[None, :] - y_true[:, None]), axis=1)))

print("\n=== Soft-label diagnostics (Human vs AI prob vector) ===")
print(f"N = {len(y_true)}")
print(f"Mean P(true human class): {mean_p_true:.4f}")
print(f"NLL (cross-entropy):     {nll:.4f}")
print(f"Brier (multiclass):      {brier:.4f}")
print(f"Ordinal MAE (E[idx]-y):  {mae:.4f}")
print(f"Ordinal RMSE:            {rmse:.4f}")
print(f"Expected |ordinal error|:{w1:.4f}")

# ---------------------------
# 5) Stratify agreement by confidence / uncertainty (optional but very informative)
# ---------------------------
tmp = df.loc[mask_soft, [HUMAN_COL, "ai_sent5_from_probs", "ai_pmax", "ai_entropy_norm"]].copy()
tmp["correct_prob_argmax"] = (tmp[HUMAN_COL] == tmp["ai_sent5_from_probs"])

# bins for pmax and entropy
tmp["pmax_bin"] = pd.cut(tmp["ai_pmax"], bins=[0, .4, .6, .8, .9, 1.0], include_lowest=True)
tmp["ent_bin"]  = pd.cut(tmp["ai_entropy_norm"], bins=[0, .2, .4, .6, .8, 1.0], include_lowest=True)

summary_pmax = tmp.groupby("pmax_bin")["correct_prob_argmax"].agg(["count", "mean"]).rename(columns={"mean":"acc"})
summary_ent  = tmp.groupby("ent_bin")["correct_prob_argmax"].agg(["count", "mean"]).rename(columns={"mean":"acc"})

print("\nAgreement vs p_max bins (Human vs prob-argmax):")
display(summary_pmax)

print("\nAgreement vs entropy bins (Human vs prob-argmax):")
display(summary_ent)


=== Human sentiment_5 vs AI hard ai_sentiment_5 ===
N = 28755
Exact agreement (accuracy): 0.5603
Quadratic weighted kappa:   0.8256

Confusion matrix (counts):


,P:very_negative,P:negative,P:neutral,P:positive,P:very_positive
T:very_negative,3002,827,109,11,4
T:negative,465,848,278,36,8
T:neutral,179,844,984,245,30
T:positive,71,373,1765,2771,718
T:very_positive,34,192,925,5530,8506



Classification report:
               precision    recall  f1-score   support

very_negative       0.80      0.76      0.78      3953
     negative       0.27      0.52      0.36      1635
      neutral       0.24      0.43      0.31      2282
     positive       0.32      0.49      0.39      5698
very_positive       0.92      0.56      0.70     15187

     accuracy                           0.56     28755
    macro avg       0.51      0.55      0.51     28755
 weighted avg       0.69      0.56      0.60     28755


=== Human sentiment_5 vs AI (ai_rating_10 -> 5-class) ===
N = 28755
Exact agreement (accuracy): 0.5603
Quadratic weighted kappa:   0.8256

Confusion matrix (counts):


,P:very_negative,P:negative,P:neutral,P:positive,P:very_positive
T:very_negative,3002,827,109,11,4
T:negative,465,848,278,36,8
T:neutral,179,844,984,245,30
T:positive,71,373,1765,2771,718
T:very_positive,34,192,925,5530,8506



Classification report:
               precision    recall  f1-score   support

very_negative       0.80      0.76      0.78      3953
     negative       0.27      0.52      0.36      1635
      neutral       0.24      0.43      0.31      2282
     positive       0.32      0.49      0.39      5698
very_positive       0.92      0.56      0.70     15187

     accuracy                           0.56     28755
    macro avg       0.51      0.55      0.51     28755
 weighted avg       0.69      0.56      0.60     28755


=== Human sentiment_5 vs AI (prob argmax) ===
N = 28755
Exact agreement (accuracy): 0.5837
Quadratic weighted kappa:   0.8168

Confusion matrix (counts):


...


Classification report:
               precision    recall  f1-score   support

very_negative       0.75      0.99      0.85      3751
     negative       0.57      0.59      0.58      3084
      neutral       0.92      0.16      0.27      4061
     positive       0.64      0.70      0.67      8593
very_positive       0.75      0.86      0.80      9266

     accuracy                           0.70     28755
    macro avg       0.73      0.66      0.63     28755
 weighted avg       0.72      0.70      0.67     28755


=== Soft-label diagnostics (Human vs AI prob vector) ===
N = 28755
Mean P(true human class): 0.4302
NLL (cross-entropy):     1.2648
Brier (multiclass):      0.5403
Ordinal MAE (E[idx]-y):  0.6782
Ordinal RMSE:            0.8278
Expected |ordinal error|:0.8145

Agreement vs p_max bins (Human vs prob-argmax):


/tmp/ipython-input-1095427977.py:172: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_pmax = tmp.groupby("pmax_bin")["correct_prob_argmax"].agg(["count", "mean"]).rename(columns={"mean":"acc"})
/tmp/ipython-input-1095427977.py:173: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_ent  = tmp.groupby("ent_bin")["correct_prob_argmax"].agg(["count", "mean"]).rename(columns={"mean":"acc"})


,count,acc
pmax_bin,,
"(-0.001, 0.4]",10927,0.348312
"(0.4, 0.6]",14129,0.696298
"(0.6, 0.8]",2805,0.844207
"(0.8, 0.9]",817,0.861689
"(0.9, 1.0]",77,0.870130



Agreement vs entropy bins (Human vs prob-argmax):


,count,acc
ent_bin,,
"(-0.001, 0.2]",77,0.870130
"(0.2, 0.4]",2898,0.863009
"(0.4, 0.6]",11281,0.759596
"(0.6, 0.8]",9108,0.439284
"(0.8, 1.0]",5391,0.305138


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, cohen_kappa_score

# --- helpers ---
LABELS_5 = ["very_negative", "negative", "neutral", "positive", "very_positive"]

def normalize_5class(x):
    """Make ai_sentiment_5 robust to casing / spaces / hyphens."""
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower().replace("-", "_").replace(" ", "_")
    # common variants (optional)
    mapping = {
        "verynegative": "very_negative",
        "verypositive": "very_positive",
        "vneg": "very_negative",
        "vpos": "very_positive",
    }
    return mapping.get(s, s)

def rating10_to_5class(r):
    """Map 1–10 rating to 5 ordinal bins: 1-2,3-4,5-6,7-8,9-10."""
    if pd.isna(r):
        return pd.NA
    try:
        rr = int(float(r))
    except Exception:
        return pd.NA
    if rr < 1 or rr > 10:
        return pd.NA
    if rr <= 2:
        return "very_negative"
    elif rr <= 4:
        return "negative"
    elif rr <= 6:
        return "neutral"
    elif rr <= 8:
        return "positive"
    else:
        return "very_positive"

# --- main ---
# df = ...  # your dataframe containing ai_sentiment_5 and ai_rating_10

df = df.copy()
df["ai_sentiment_5_norm"] = df["ai_sentiment_5"].apply(normalize_5class)
df["ai_sent5_from_rating10"] = df["ai_rating_10"].apply(rating10_to_5class)

sub = df[["ai_sentiment_5_norm", "ai_sent5_from_rating10"]].dropna()

y1 = sub["ai_sentiment_5_norm"]
y2 = sub["ai_sent5_from_rating10"]

acc = accuracy_score(y1, y2)
kappa_q = cohen_kappa_score(y1, y2, labels=LABELS_5, weights="quadratic")
cm = confusion_matrix(y1, y2, labels=LABELS_5)

print(f"N = {len(sub)}")
print(f"Exact agreement (accuracy): {acc:.4f}")
print(f"Quadratic weighted kappa:   {kappa_q:.4f}\n")

print("Confusion matrix (counts):")
print(pd.DataFrame(cm, index=[f"T:{c}" for c in LABELS_5], columns=[f"P:{c}" for c in LABELS_5]))
print("\nClassification report:")
print(classification_report(y1, y2, labels=LABELS_5, digits=2))

N = 28755
Exact agreement (accuracy): 1.0000
Quadratic weighted kappa:   1.0000

Confusion matrix (counts):
                 P:very_negative  P:negative  P:neutral  P:positive  \
T:very_negative             3751           0          0           0   
T:negative                     0        3084          0           0   
T:neutral                      0           0       4061           0   
T:positive                     0           0          0        8593   
T:very_positive                0           0          0           0   

                 P:very_positive  
T:very_negative                0  
T:negative                     0  
T:neutral                      0  
T:positive                     0  
T:very_positive             9266  

Classification report:
               precision    recall  f1-score   support

very_negative       1.00      1.00      1.00      3751
     negative       1.00      1.00      1.00      3084
      neutral       1.00      1.00      1.00      4061
     posit

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score

# ---------------------------
# REQUIREMENTS from your existing pipeline:
# df contains:
#   HUMAN_COL (sentiment_5)
#   AI_HARD_COL (ai_sentiment_5)
#   ai_sent5_from_probs (prob argmax label, 5-class)
#   ai_pmax (max prob)
# ---------------------------

ORDER5 = ["very_negative", "negative", "neutral", "positive", "very_positive"]
VAL5 = {"very_negative": -2, "negative": -1, "neutral": 0, "positive": 1, "very_positive": 2}

def norm5(x):
    if pd.isna(x):
        return np.nan
    return (str(x).strip().lower().replace(" ", "_").replace("-", "_"))

def label_distribution(s, order=ORDER5):
    s = s.dropna().astype(str)
    counts = s.value_counts().reindex(order).fillna(0).astype(int)
    props = counts / counts.sum() if counts.sum() > 0 else counts.astype(float)
    return pd.DataFrame({"count": counts, "prop": props})

def jensen_shannon(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))
    return float(np.sqrt(0.5 * (kl_pm + kl_qm)))

def ordinal_error_summary(y_true, y_pred):
    yt = y_true.map(norm5)
    yp = y_pred.map(norm5)
    m = yt.notna() & yp.notna()
    yt = yt[m].astype(str)
    yp = yp[m].astype(str)

    yv = yt.map(VAL5).to_numpy()
    pv = yp.map(VAL5).to_numpy()

    abs_err = np.abs(pv - yv)
    out = {
        "N": int(m.sum()),
        "mean_abs_ord_err": float(abs_err.mean()),
        "ord_rmse": float(np.sqrt(np.mean((pv - yv)**2))),
        "within_1_class": float((abs_err <= 1).mean()),
    }
    return out

def bootstrap_ci_acc_qwk(y_true, y_pred, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    yt = y_true.map(norm5)
    yp = y_pred.map(norm5)
    m = yt.notna() & yp.notna()
    yt = yt[m].astype(str).to_numpy()
    yp = yp[m].astype(str).to_numpy()
    n = len(yt)

    accs = np.empty(B, dtype=float)
    qwks = np.empty(B, dtype=float)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        ytb, ypb = yt[idx], yp[idx]
        accs[b] = accuracy_score(ytb, ypb)
        qwks[b] = cohen_kappa_score(ytb, ypb, labels=ORDER5, weights="quadratic")

    def ci(x):
        return (float(np.quantile(x, 0.025)), float(np.quantile(x, 0.975)))

    return {
        "N": n,
        "acc_mean": float(accs.mean()),
        "acc_ci95": ci(accs),
        "qwk_mean": float(qwks.mean()),
        "qwk_ci95": ci(qwks),
    }

def ece_from_pmax(y_true, y_pred, pmax, n_bins=10):
    y_true = np.asarray([norm5(x) for x in y_true])
    y_pred = np.asarray([norm5(x) for x in y_pred])
    pmax = np.asarray(pmax, dtype=float)

    m = pd.notna(y_true) & pd.notna(y_pred) & pd.notna(pmax)
    y_true = y_true[m]
    y_pred = y_pred[m]
    pmax = pmax[m]

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    rows = []

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        sel = (pmax > lo) & (pmax <= hi) if i > 0 else (pmax >= lo) & (pmax <= hi)
        if sel.sum() == 0:
            rows.append([f"({lo:.1f},{hi:.1f}]", 0, np.nan, np.nan, np.nan])
            continue
        acc_b = float((y_true[sel] == y_pred[sel]).mean())
        conf_b = float(pmax[sel].mean())
        w = float(sel.mean())
        ece += w * abs(acc_b - conf_b)
        rows.append([f"({lo:.1f},{hi:.1f}]", int(sel.sum()), acc_b, conf_b, abs(acc_b - conf_b)])

    tab = pd.DataFrame(rows, columns=["bin", "count", "acc", "mean_conf", "|acc-conf|"])
    return float(ece), tab

# ---------------------------
# IMPLEMENTATION (runs now)
# ---------------------------
HUMAN_COL = "sentiment_5"
AI_HARD_COL = "ai_sentiment_5"
AI_ARGMAX_COL = "ai_sent5_from_probs"
PMAX_COL = "ai_pmax"

# Normalize once
df[HUMAN_COL] = df[HUMAN_COL].map(norm5)
df[AI_HARD_COL] = df[AI_HARD_COL].map(norm5)
df[AI_ARGMAX_COL] = df[AI_ARGMAX_COL].map(norm5)

print("\n==============================")
print("EXTRA LABEL EVAL DIAGNOSTICS")
print("==============================")

# (A) Marginal distributions + JS distance
for pred_col, name in [(AI_HARD_COL, "AI hard"), (AI_ARGMAX_COL, "AI prob-argmax")]:
    tab_h = label_distribution(df[HUMAN_COL])
    tab_a = label_distribution(df[pred_col])
    js = jensen_shannon(tab_h["prop"].values, tab_a["prop"].values)

    print(f"\n--- Marginal shift: Human vs {name} ---")
    display(pd.concat({"Human": tab_h, name: tab_a}, axis=1))
    print("JS distance:", round(js, 4))

# (B) Ordinal-distance error for hard labels (interpretable)
for pred_col, name in [(AI_HARD_COL, "Human vs AI hard"),
                       (AI_ARGMAX_COL, "Human vs AI prob-argmax")]:
    out = ordinal_error_summary(df[HUMAN_COL], df[pred_col])
    print(f"\n--- Ordinal error: {name} ---")
    for k, v in out.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# (C) Bootstrap CI for accuracy + QWK (hard agreement)
for pred_col, name in [(AI_HARD_COL, "Human vs AI hard"),
                       (AI_ARGMAX_COL, "Human vs AI prob-argmax")]:
    ci = bootstrap_ci_acc_qwk(df[HUMAN_COL], df[pred_col], B=2000, seed=123)
    print(f"\n--- Bootstrap 95% CI: {name} ---")
    print("N:", ci["N"])
    print(f"Accuracy: {ci['acc_mean']:.4f}  (95% CI {ci['acc_ci95'][0]:.4f}, {ci['acc_ci95'][1]:.4f})")
    print(f"QWK:      {ci['qwk_mean']:.4f}  (95% CI {ci['qwk_ci95'][0]:.4f}, {ci['qwk_ci95'][1]:.4f})")

# (D) Calibration via ECE (for prob-argmax vs human)
if PMAX_COL in df.columns:
    mask = df[HUMAN_COL].notna() & df[AI_ARGMAX_COL].notna() & df[PMAX_COL].notna()
    ece, tab = ece_from_pmax(df.loc[mask, HUMAN_COL],
                             df.loc[mask, AI_ARGMAX_COL],
                             df.loc[mask, PMAX_COL],
                             n_bins=10)
    print("\n--- Calibration: ECE (prob-argmax using pmax) ---")
    print("ECE:", round(ece, 4))
    display(tab)
else:
    print("\n[Skip] ai_pmax not found; cannot compute ECE.")


EXTRA LABEL EVAL DIAGNOSTICS

--- Marginal shift: Human vs AI hard ---


Human           AI hard          
               count      prop   count      prop
very_negative   3953  0.137472    3751  0.130447
negative        1635  0.056860    3084  0.107251
neutral         2282  0.079360    4061  0.141228
positive        5698  0.198157    8593  0.298835
very_positive  15187  0.528152    9266  0.322240

JS distance: 0.1615

--- Marginal shift: Human vs AI prob-argmax ---


Human           AI prob-argmax          
               count      prop          count      prop
very_negative   3953  0.137472           4987  0.173431
negative        1635  0.056860           3174  0.110381
neutral         2282  0.079360            684  0.023787
positive        5698  0.198157           9341  0.324848
very_positive  15187  0.528152          10569  0.367553

JS distance: 0.1682

--- Ordinal error: Human vs AI hard ---
N: 28755
mean_abs_ord_err: 0.5207
ord_rmse: 0.8428
within_1_class: 0.9314

--- Ordinal error: Human vs AI prob-argmax ---
N: 28755
mean_abs_ord_err: 0.5253
ord_rmse: 0.8984
within_1_class: 0.9190

--- Bootstrap 95% CI: Human vs AI hard ---
N: 28755
Accuracy: 0.5603  (95% CI 0.5545, 0.5659)
QWK:      0.8256  (95% CI 0.8213, 0.8301)

--- Bootstrap 95% CI: Human vs AI prob-argmax ---
N: 28755
Accuracy: 0.5836  (95% CI 0.5779, 0.5896)
QWK:      0.8168  (95% CI 0.8117, 0.8218)

--- Calibration: ECE (prob-argmax using pmax) ---
ECE: 0.1033


,bin,count,acc,mean_conf,|acc-conf|
0,"(0.0,0.1]",0,NaN,NaN,NaN
1,"(0.1,0.2]",0,NaN,NaN,NaN
2,"(0.2,0.3]",2368,0.243666,0.299716,0.056050
3,"(0.3,0.4]",8559,0.377264,0.390724,0.013460
4,"(0.4,0.5]",9837,0.645522,0.494109,0.151413
5,"(0.5,0.6]",4292,0.812675,0.599026,0.213648
6,"(0.6,0.7]",1442,0.818308,0.695791,0.122517
7,"(0.7,0.8]",1363,0.871607,0.797891,0.073716
8,"(0.8,0.9]",817,0.861689,0.898639,0.036950
9,"(0.9,1.0]",77,0.870130,1.000000,0.129870


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---------------------------
# Helpers
# ---------------------------
ORDER5 = ["very_negative", "negative", "neutral", "positive", "very_positive"]
PROB_COLS_DEFAULT = [
    "ai_prob_very_negative",
    "ai_prob_negative",
    "ai_prob_neutral",
    "ai_prob_positive",
    "ai_prob_very_positive",
]

def norm5(x):
    if pd.isna(x):
        return np.nan
    return (str(x).strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_"))

def defensive_renorm(P, eps=1e-12):
    P = np.clip(P, eps, 1.0)
    rs = P.sum(axis=1, keepdims=True)
    return P / rs

def entropy_norm(P):
    # normalized Shannon entropy in [0,1]
    K = P.shape[1]
    H = -np.sum(P * np.log(P), axis=1)
    return H / np.log(K)

def fit_disagreement_logit(
    df,
    human_col="sentiment_5",
    ai_col="ai_sentiment_5",
    review_col="review",
    rating_col="rating",
    prob_cols=PROB_COLS_DEFAULT,
    use_entropy=True,                 # if False, uses pmax instead
    add_eff_saf=False,                # optionally add efficacy/safety if present
    robust="HC3",                     # "HC3" (default) or None
    cluster_col=None,                 # e.g., "drugName" if you want cluster-robust SEs
):
    d = df.copy()

    # --- normalize labels ---
    d[human_col] = d[human_col].map(norm5)
    d[ai_col] = d[ai_col].map(norm5)

    # --- keep rows with both labels ---
    mask = d[human_col].notna() & d[ai_col].notna()
    d = d.loc[mask].copy()

    # --- outcome: disagreement indicator ---
    d["disagree"] = (d[human_col] != d[ai_col]).astype(int)

    # --- review length ---
    if review_col in d.columns:
        d["review_len"] = d[review_col].astype(str).str.len()
    else:
        d["review_len"] = np.nan

    # --- rating numeric ---
    if rating_col in d.columns:
        d["rating_num"] = pd.to_numeric(d[rating_col], errors="coerce")
    else:
        d["rating_num"] = np.nan

    # --- uncertainty features from probabilities ---
    missing = [c for c in prob_cols if c not in d.columns]
    if missing:
        raise ValueError(f"Missing prob columns needed for uncertainty: {missing}")

    P = d[prob_cols].astype(float).to_numpy()
    P = defensive_renorm(P)
    d["pmax"] = P.max(axis=1)
    d["entropy_norm"] = entropy_norm(P)

    # --- choose uncertainty regressor ---
    unc = "entropy_norm" if use_entropy else "pmax"

    # --- build formula ---
    base_terms = [unc, "review_len", "rating_num"]

    # optionally add efficacy/safety as categorical controls if columns exist
    if add_eff_saf:
        for col in ["ai_efficacy", "ai_safety"]:
            if col in d.columns:
                base_terms.append(f"C({col})")

    # drop rows with missing covariates used in model
    needed_cols = ["disagree", unc, "review_len", "rating_num"]
    d_model = d.dropna(subset=needed_cols).copy()

    formula = "disagree ~ " + " + ".join(base_terms)

    # --- fit logit ---
    model = smf.logit(formula, data=d_model)
    if cluster_col is not None and cluster_col in d_model.columns:
        res = model.fit(disp=False, cov_type="cluster", cov_kwds={"groups": d_model[cluster_col]})
    elif robust is not None:
        res = model.fit(disp=False, cov_type=robust)
    else:
        res = model.fit(disp=False)

    return res, formula, d_model


def summarize_odds_ratios(res):
    """Pretty OR table with 95% CI."""
    params = res.params
    se = res.bse
    z = 1.96
    out = pd.DataFrame({
        "coef": params,
        "se": se,
        "OR": np.exp(params),
        "OR_2.5%": np.exp(params - z*se),
        "OR_97.5%": np.exp(params + z*se),
        "p": res.pvalues
    })
    return out.sort_values("p")


# ---------------------------
# IMPLEMENTATION ON df
# ---------------------------

# 1) Disagreement ~ entropy + controls
res_ent, formula_ent, d_used_ent = fit_disagreement_logit(
    df,
    human_col="sentiment_5",
    ai_col="ai_sentiment_5",
    review_col="review",
    rating_col="rating",
    use_entropy=True,
    add_eff_saf=False,   # flip to True if you want to control for ai_efficacy/ai_safety
    robust="HC3",
    cluster_col=None,    # e.g. "drugName" if you want clustered SE
)
print("MODEL (entropy):", formula_ent)
print(res_ent.summary())
display(summarize_odds_ratios(res_ent).head(15))

# 2) Disagreement ~ pmax + controls
res_pmax, formula_pmax, d_used_pmax = fit_disagreement_logit(
    df,
    human_col="sentiment_5",
    ai_col="ai_sentiment_5",
    review_col="review",
    rating_col="rating",
    use_entropy=False,   # uses pmax
    add_eff_saf=False,
    robust="HC3",
    cluster_col=None,
)
print("\nMODEL (pmax):", formula_pmax)
print(res_pmax.summary())
display(summarize_odds_ratios(res_pmax).head(15))

MODEL (entropy): disagree ~ entropy_norm + review_len + rating_num
                           Logit Regression Results                           
Dep. Variable:               disagree   No. Observations:                28755
Model:                          Logit   Df Residuals:                    28751
Method:                           MLE   Df Model:                            3
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                  0.1831
Time:                        23:46:17   Log-Likelihood:                -16110.
converged:                       True   LL-Null:                       -19722.
Covariance Type:                  HC3   LLR p-value:                     0.000
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -5.8766      0.087    -67.572      0.000      -6.047      -5.706
entropy_norm     6.8212      0.098     69.617      0.000  

,coef,se,OR,OR_2.5%,OR_97.5%,p
Intercept,-5.876569,0.086967,0.002804,0.002365,0.003326,0.000000e+00
entropy_norm,6.821213,0.097982,917.096863,756.852753,1111.268543,0.000000e+00
rating_num,0.182461,0.004833,1.200168,1.188853,1.211590,0.000000e+00
review_len,-0.000661,0.000058,0.999339,0.999225,0.999453,6.404591e-30



MODEL (pmax): disagree ~ pmax + review_len + rating_num
                           Logit Regression Results                           
Dep. Variable:               disagree   No. Observations:                28755
Model:                          Logit   Df Residuals:                    28751
Method:                           MLE   Df Model:                            3
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                  0.1787
Time:                        23:46:17   Log-Likelihood:                -16197.
converged:                       True   LL-Null:                       -19722.
Covariance Type:                  HC3   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.3783      0.077     43.827      0.000       3.227       3.529
pmax          -9.4223      0.150    -62.651      0.000      -9.717      -9

,coef,se,OR,OR_2.5%,OR_97.5%,p
Intercept,3.378319,0.077083,29.321448,25.209898,34.103562,0.000000e+00
pmax,-9.422337,0.150394,0.000081,0.000060,0.000109,0.000000e+00
rating_num,0.159290,0.004620,1.172678,1.162106,1.183346,1.878604e-260
review_len,-0.000504,0.000057,0.999496,0.999385,0.999608,9.461279e-19


In [ ]:
import numpy as np
import pandas as pd

# ---- config ----
HUMAN_RATING = "rating"          # original user rating column (string/int)
HUMAN5 = "sentiment_5"           # human 5-class
AI5 = "ai_sentiment_5"           # AI hard 5-class
AI_R10 = "ai_rating_10"          # AI 1..10

def norm5(x):
    if pd.isna(x): return np.nan
    return str(x).strip().lower().replace(" ", "_").replace("-", "_")

def rating10_to_sent5(r):
    if pd.isna(r): return np.nan
    r = int(r)
    if r <= 2:  return "very_negative"
    if r <= 4:  return "negative"
    if r <= 6:  return "neutral"
    if r <= 8:  return "positive"
    return "very_positive"

# =========================
# 1) Check human rating_num
# =========================
df["rating_num"] = pd.to_numeric(df[HUMAN_RATING], errors="coerce")

print("Human rating_num summary:")
print(df["rating_num"].describe())
print("\nHuman rating_num invalid (NaN or outside 1..10):",
      int(df["rating_num"].isna().sum() + ((df["rating_num"]<1)|(df["rating_num"]>10)).sum()))
print("\nTop raw values of `rating` (to confirm it's the original column):")
print(df[HUMAN_RATING].value_counts(dropna=False).head(15))

# If you have a `date` column and want to check parsing issues:
if "date" in df.columns:
    print("\nDate col dtype:", df["date"].dtype)

# ======================
# 2) Check AI ai_rating_10
# ======================
df["ai_rating10_num"] = pd.to_numeric(df[AI_R10], errors="coerce")

print("\nAI ai_rating_10 summary:")
print(df["ai_rating10_num"].describe())
print("\nAI ai_rating_10 invalid (NaN or outside 1..10):",
      int(df["ai_rating10_num"].isna().sum() + ((df["ai_rating10_num"]<1)|(df["ai_rating10_num"]>10)).sum()))
print("\nTop raw values of `ai_rating_10`:")
print(df[AI_R10].value_counts(dropna=False).head(15))

# ==========================================
# 3) Check ai_sentiment_5 <-> ai_rating_10
# ==========================================
df["ai_sent5_from_rating10"] = df["ai_rating10_num"].apply(rating10_to_sent5)
df["ai_sentiment_5_clean"] = df[AI5].map(norm5)

mask = df["ai_sent5_from_rating10"].notna() & df["ai_sentiment_5_clean"].notna()
match_rate = (df.loc[mask, "ai_sent5_from_rating10"] == df.loc[mask, "ai_sentiment_5_clean"]).mean()

print("\nAI hard (ai_sentiment_5) vs binned ai_rating_10 match rate:", round(match_rate, 6))
if match_rate < 1.0:
    bad = df.loc[mask].copy()
    bad = bad[bad["ai_sent5_from_rating10"] != bad["ai_sentiment_5_clean"]]
    print("Mismatches:", len(bad))
    display(bad[[AI_R10, "ai_sent5_from_rating10", AI5, "review"]].head(20))

# ==========================================
# 4) Confirm disagreement outcome definition
# ==========================================
df["human_sent5_clean"] = df[HUMAN5].map(norm5)
mask2 = df["human_sent5_clean"].notna() & df["ai_sentiment_5_clean"].notna()

df["disagree"] = (df["human_sent5_clean"] != df["ai_sentiment_5_clean"]).astype(int)

print("\nDisagreement rate (human 5 vs AI hard 5):",
      df.loc[mask2, "disagree"].mean().round(4),
      "| N =", int(mask2.sum()))

# quick crosstab sanity
print("\nCrosstab (human vs AI hard) top-left sanity:")
display(pd.crosstab(df.loc[mask2, "human_sent5_clean"], df.loc[mask2, "ai_sentiment_5_clean"]).reindex(
    index=["very_negative","negative","neutral","positive","very_positive"],
    columns=["very_negative","negative","neutral","positive","very_positive"]
))

Human rating_num summary:
count    28755.000000
mean         7.407060
std          3.037296
min          1.000000
25%          6.000000
50%          9.000000
75%         10.000000
max         10.000000
Name: rating_num, dtype: float64

Human rating_num invalid (NaN or outside 1..10): 0

Top raw values of `rating` (to confirm it's the original column):
rating
10    9444
9     5743
8     3873
1     2944
7     1825
5     1163
6     1119
2     1009
3      927
4      708
Name: count, dtype: int64

Date col dtype: object

AI ai_rating_10 summary:
count    28755.000000
mean         6.657938
std          2.713483
min          1.000000
25%          5.000000
50%          8.000000
75%          9.000000
max         10.000000
Name: ai_rating10_num, dtype: float64

AI ai_rating_10 invalid (NaN or outside 1..10): 0

Top raw values of `ai_rating_10`:
ai_rating_10
9.0     6131
8.0     5193
7.0     3400
10.0    3135
6.0     2392
2.0     2292
5.0     1669
4.0     1584
3.0     1500
1.0     1459
Name: coun

ai_sentiment_5_clean,very_negative,negative,neutral,positive,very_positive
human_sent5_clean,,,,,
very_negative,3002,827,109,11,4
negative,465,848,278,36,8
neutral,179,844,984,245,30
positive,71,373,1765,2771,718
very_positive,34,192,925,5530,8506


In [ ]:
ct = pd.crosstab(df.loc[mask2, "human_sent5_clean"], df.loc[mask2, "ai_sentiment_5_clean"])
ct = ct.reindex(index=["very_negative","negative","neutral","positive","very_positive"],
                columns=["very_negative","negative","neutral","positive","very_positive"],
                fill_value=0)
display(ct)

ai_sentiment_5_clean,very_negative,negative,neutral,positive,very_positive
human_sent5_clean,,,,,
very_negative,3002,827,109,11,4
negative,465,848,278,36,8
neutral,179,844,984,245,30
positive,71,373,1765,2771,718
very_positive,34,192,925,5530,8506


In [ ]:
PROB_COLS = ["ai_prob_very_negative","ai_prob_negative","ai_prob_neutral","ai_prob_positive","ai_prob_very_positive"]
P = df[PROB_COLS].astype(float)

# any negatives?
print("Any prob < 0:", (P < 0).any().any())

row_sum = P.sum(axis=1)
print("Row-sum summary:")
print(row_sum.describe())
print("Rows far from 1.0 (abs > 1e-3):", int((row_sum.sub(1).abs() > 1e-3).sum()))

Any prob < 0: False
Row-sum summary:
count    2.875500e+04
mean     1.000000e+00
std      1.293432e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
Rows far from 1.0 (abs > 1e-3): 0


In [ ]:
pmax = P.max(axis=1)
entropy = -(P.clip(1e-12,1).to_numpy() * np.log(P.clip(1e-12,1).to_numpy())).sum(axis=1) / np.log(5)

print("corr(entropy_norm, pmax) =", np.corrcoef(entropy, pmax)[0,1])

corr(entropy_norm, pmax) = -0.9329342717751969


In [ ]:
import numpy as np
import pandas as pd

HUMAN_COL   = "sentiment_5"
AI_HARD_COL = "ai_sentiment_5"

PROB_COLS = [
    "ai_prob_very_negative",
    "ai_prob_negative",
    "ai_prob_neutral",
    "ai_prob_positive",
    "ai_prob_very_positive",
]

def norm5(x):
    if pd.isna(x): return np.nan
    return (str(x).strip().lower().replace(" ", "_").replace("-", "_"))

def defensive_renorm(p, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    return p / p.sum(axis=1, keepdims=True)

def entropy_norm(p):
    K = p.shape[1]
    H = -np.sum(p * np.log(p), axis=1)
    return H / np.log(K)

def make_df2(df):
    df2 = df.copy()

    # --- clean labels ---
    df2["human_sent5_clean"] = df2[HUMAN_COL].map(norm5)
    df2["ai_sentiment_5_clean"] = df2[AI_HARD_COL].map(norm5)

    # --- outcome: disagree (hard vs human) ---
    mask = df2["human_sent5_clean"].notna() & df2["ai_sentiment_5_clean"].notna()
    df2.loc[mask, "disagree"] = (df2.loc[mask, "human_sent5_clean"] != df2.loc[mask, "ai_sentiment_5_clean"]).astype(int)

    # --- predictors: review length ---
    if "review" in df2.columns:
        df2["review_len"] = df2["review"].astype(str).str.len()
    else:
        df2["review_len"] = np.nan

    # --- predictors: rating numeric (from human rating column "rating") ---
    # If you already have rating_num, this will just overwrite consistently.
    if "rating_num" in df2.columns:
        df2["rating_num"] = pd.to_numeric(df2["rating_num"], errors="coerce")
    elif "rating" in df2.columns:
        df2["rating_num"] = pd.to_numeric(df2["rating"], errors="coerce")
    else:
        df2["rating_num"] = np.nan

    # --- predictors: pmax + entropy_norm from probability vector ---
    missing = [c for c in PROB_COLS if c not in df2.columns]
    if len(missing) > 0:
        raise KeyError(f"Missing probability columns needed for entropy/pmax: {missing}")

    p_raw = df2[PROB_COLS].astype(float).to_numpy()
    p = defensive_renorm(p_raw)

    df2["pmax"] = p.max(axis=1)
    df2["entropy_norm"] = entropy_norm(p)

    return df2

df2 = make_df2(df)
print("N (with disagree defined):", int(df2["disagree"].notna().sum()))
print("Disagree rate:", float(df2.loc[df2["disagree"].notna(), "disagree"].mean()))

N (with disagree defined): 28755
Disagree rate: 0.4397148322030951


In [ ]:
# ============================================================
# FULL DROP-IN CELL: Rescaled Logit + robust SE via cov_type="HC3"
# - ORs are per 0.1 increase in entropy/pmax
# - ORs are per 100 characters increase in review length
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ---------------------------
# CONFIG (edit if needed)
# ---------------------------
HUMAN_COL   = "sentiment_5"
AI_HARD_COL = "ai_sentiment_5"
REVIEW_COL  = "review"
RATING_COL  = "rating"

PROB_COLS = [
    "ai_prob_very_negative",
    "ai_prob_negative",
    "ai_prob_neutral",
    "ai_prob_positive",
    "ai_prob_very_positive",
]

# ---------------------------
# HELPERS
# ---------------------------
def norm5(x):
    if pd.isna(x):
        return np.nan
    return (str(x).strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_"))

def defensive_renorm(p, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    s = p.sum(axis=1, keepdims=True)
    return p / s

def entropy_norm(p):
    K = p.shape[1]
    H = -np.sum(p * np.log(p), axis=1)
    return H / np.log(K)

def build_df2(df):
    df2 = df.copy()

    # clean labels
    df2["human_sent5_clean"] = df2[HUMAN_COL].map(norm5)
    df2["ai_sent5_clean"]    = df2[AI_HARD_COL].map(norm5)

    # outcome: disagree (based on HUMAN vs AI HARD labels)
    mask = df2["human_sent5_clean"].notna() & df2["ai_sent5_clean"].notna()
    df2.loc[mask, "disagree"] = (
        df2.loc[mask, "human_sent5_clean"] != df2.loc[mask, "ai_sent5_clean"]
    ).astype(int)

    # review length
    if REVIEW_COL in df2.columns:
        df2["review_len"] = df2[REVIEW_COL].astype(str).str.len()
    else:
        df2["review_len"] = np.nan

    # rating numeric
    if "rating_num" in df2.columns:
        df2["rating_num"] = pd.to_numeric(df2["rating_num"], errors="coerce")
    elif RATING_COL in df2.columns:
        df2["rating_num"] = pd.to_numeric(df2[RATING_COL], errors="coerce")
    else:
        df2["rating_num"] = np.nan

    # probs -> pmax + entropy_norm
    missing = [c for c in PROB_COLS if c not in df2.columns]
    if missing:
        raise KeyError(f"Missing probability columns: {missing}")

    p_raw = df2[PROB_COLS].astype(float).to_numpy()
    p = defensive_renorm(p_raw)

    df2["pmax"] = p.max(axis=1)
    df2["entropy_norm"] = entropy_norm(p)

    # ---------------------------
    # RESCALES for interpretability
    # ---------------------------
    # OR per 0.1 increase (instead of per 1.0)
    df2["entropy_per_0p1"] = df2["entropy_norm"] / 0.1
    df2["pmax_per_0p1"]    = df2["pmax"] / 0.1

    # OR per 100 characters (instead of per 1 char)
    df2["review_len_per_100"] = df2["review_len"] / 100.0

    return df2

def fit_disagree_logit(df2, predictor, cov_type="HC3"):
    """
    Fit: disagree ~ predictor + review_len_per_100 + rating_num
    """
    needed = ["disagree", predictor, "review_len_per_100", "rating_num"]
    d = df2[needed].dropna().copy()
    d["disagree"] = d["disagree"].astype(int)

    formula = f"disagree ~ {predictor} + review_len_per_100 + rating_num"
    res = smf.logit(formula, data=d).fit(disp=False, cov_type=cov_type)
    return res, d

def or_table_from_result(res, alpha=0.05):
    b = res.params
    se = res.bse
    p = res.pvalues
    ci = res.conf_int(alpha=alpha)
    ci.columns = ["coef_2.5%", "coef_97.5%"]

    out = pd.DataFrame({
        "coef": b,
        "se": se,
        "OR": np.exp(b),
        "OR_2.5%": np.exp(ci["coef_2.5%"]),
        "OR_97.5%": np.exp(ci["coef_97.5%"]),
        "p": p,
    })
    return out

# ============================================================
# RUN (ASSUMES df exists)
# ============================================================

df2 = build_df2(df)

print("N (with disagree defined):", int(df2["disagree"].notna().sum()))
print("Disagree rate:", float(df2.loc[df2["disagree"].notna(), "disagree"].mean()))

# ---- Model 1 (RESCALED): entropy_per_0p1 ----
res_ent, _ = fit_disagree_logit(df2, predictor="entropy_per_0p1", cov_type="HC3")
print("\nMODEL (entropy, rescaled): disagree ~ entropy_per_0p1 + review_len_per_100 + rating_num")
print(res_ent.summary())
display(or_table_from_result(res_ent))

# ---- Model 2 (RESCALED): pmax_per_0p1 ----
res_pmax, _ = fit_disagree_logit(df2, predictor="pmax_per_0p1", cov_type="HC3")
print("\nMODEL (pmax, rescaled): disagree ~ pmax_per_0p1 + review_len_per_100 + rating_num")
print(res_pmax.summary())
display(or_table_from_result(res_pmax))

N (with disagree defined): 28755
Disagree rate: 0.4397148322030951

MODEL (entropy, rescaled): disagree ~ entropy_per_0p1 + review_len_per_100 + rating_num
                           Logit Regression Results                           
Dep. Variable:               disagree   No. Observations:                28755
Model:                          Logit   Df Residuals:                    28751
Method:                           MLE   Df Model:                            3
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                  0.1831
Time:                        23:46:18   Log-Likelihood:                -16110.
converged:                       True   LL-Null:                       -19722.
Covariance Type:                  HC3   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -5.8766      0.0

,coef,se,OR,OR_2.5%,OR_97.5%,p
Intercept,-5.876569,0.086967,0.002804,0.002365,0.003326,0.000000e+00
entropy_per_0p1,0.682121,0.009798,1.978069,1.940445,2.016423,0.000000e+00
review_len_per_100,-0.066084,0.005816,0.936052,0.925443,0.946783,6.404591e-30
rating_num,0.182461,0.004833,1.200168,1.188854,1.211590,0.000000e+00



MODEL (pmax, rescaled): disagree ~ pmax_per_0p1 + review_len_per_100 + rating_num
                           Logit Regression Results                           
Dep. Variable:               disagree   No. Observations:                28755
Model:                          Logit   Df Residuals:                    28751
Method:                           MLE   Df Model:                            3
Date:                Sat, 20 Dec 2025   Pseudo R-squ.:                  0.1787
Time:                        23:46:18   Log-Likelihood:                -16197.
converged:                       True   LL-Null:                       -19722.
Covariance Type:                  HC3   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              3.3783      0.077     43.827      0.000       3.227       3.529
pmax_per_0p1          -0

,coef,se,OR,OR_2.5%,OR_97.5%,p
Intercept,3.378319,0.077083,29.321448,25.209968,34.103468,0.000000e+00
pmax_per_0p1,-0.942234,0.015039,0.389756,0.378435,0.401416,0.000000e+00
review_len_per_100,-0.050377,0.005698,0.950871,0.940311,0.961550,9.461279e-19
rating_num,0.159290,0.004620,1.172678,1.162106,1.183346,1.878604e-260


In [2]:
import numpy as np
import pandas as pd

# ----------------------------
# Utilities
# ----------------------------
def make_review_key(df, review_col="review", clean_ws=True, key_col="review_key"):
    d = df.copy()
    if review_col not in d.columns:
        raise KeyError(f"review_col='{review_col}' not found. Available cols: {list(d.columns)[:30]} ...")
    s = d[review_col].fillna("").astype(str)
    if clean_ws:
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    d[key_col] = s
    return d

def guess_label_cols(df):
    # tries to find plausible label columns automatically
    candidates = []
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in ["label", "rating", "score", "sentiment", "y_", "target"]):
            candidates.append(c)
    # remove obviously-non-label columns
    bad = set(["split", "id", "review", "text", "text_orig"])
    candidates = [c for c in candidates if c.lower() not in bad]
    return candidates

def _maj_share(s):
    vc = s.value_counts(dropna=False)
    return float(vc.iloc[0] / vc.sum()) if len(vc) else np.nan

def _entropy(s):
    vc = s.value_counts(dropna=False, normalize=True)
    p = vc.values
    return float(-(p * np.log(p + 1e-12)).sum())

# ----------------------------
# 1) Duplicate label consistency
# ----------------------------
def dup_label_consistency(
    df,
    review_col="review",
    label_cols=None,
    clean_ws=True,
    key_col="review_key",
):
    d = make_review_key(df, review_col=review_col, clean_ws=clean_ws, key_col=key_col)

    if label_cols is None:
        label_cols = guess_label_cols(d)
        if not label_cols:
            raise ValueError(
                "Could not auto-detect label columns. Pass label_cols=[...] explicitly."
            )

    missing = [c for c in label_cols if c not in d.columns]
    if missing:
        raise KeyError(f"label_cols missing in df: {missing}. Available cols: {list(d.columns)[:30]} ...")

    g = d.groupby(key_col, dropna=False)
    out = pd.DataFrame({"n": g.size()})

    for col in label_cols:
        out[f"{col}_nunique"] = g[col].nunique(dropna=False)
        out[f"{col}_majshare"] = g[col].apply(_maj_share)
        out[f"{col}_entropy"]  = g[col].apply(_entropy)

    out = out.reset_index()
    dup = out[out["n"] >= 2].copy()
    return d, out, dup, label_cols

# ----------------------------
# 2) Text-only ceiling for each label source
# ----------------------------
def text_only_ceiling(df, key_col, label_col):
    g = df.groupby(key_col, dropna=False)[label_col]
    best_correct = g.apply(lambda s: s.value_counts(dropna=False).max()).sum()
    return float(best_correct / len(df))

# ----------------------------
# 3) Majority label agreement between two sources (per unique text)
# ----------------------------
def group_majority(df, key_col, label_col):
    return df.groupby(key_col, dropna=False)[label_col].apply(
        lambda s: s.value_counts(dropna=False).idxmax()
    )

def majority_agreement(df, key_col, label_a, label_b):
    a = group_majority(df, key_col, label_a).rename(label_a)
    b = group_majority(df, key_col, label_b).rename(label_b)
    maj = pd.concat([a, b], axis=1).dropna()
    return float((maj[label_a] == maj[label_b]).mean()), int(len(maj))

# ----------------------------
# 4) Metadata variation within duplicate texts (optional)
# ----------------------------
def dup_metadata_variation(df, key_col, drug_col="drug_id", cond_col="cond_id", useful_col="useful_z"):
    for c in [drug_col, cond_col, useful_col]:
        if c not in df.columns:
            raise KeyError(f"'{c}' not found in df. Available cols: {list(df.columns)[:30]} ...")

    g = df.groupby(key_col, dropna=False)
    out = g.agg(
        n=(key_col, "size"),
        drug_nunique=(drug_col, "nunique"),
        cond_nunique=(cond_col, "nunique"),
        useful_nunique=(useful_col, "nunique"),
    ).reset_index()

    out = out[out["n"] >= 2].copy()
    out["drug_or_cond_varies"] = (out["drug_nunique"] > 1) | (out["cond_nunique"] > 1)
    return out

# ==========================================================
# RUN THIS SECTION (edit only review_col / label_cols if needed)
# ==========================================================

# 1) Build keys + duplicate consistency table
d, all_groups, dup_groups, detected_labels = dup_label_consistency(
    df,
    review_col="review",       # <-- change if your column isn't named "review"
    label_cols=None,           # <-- OR set explicitly, e.g. ["human_label","gpt_text_label","gpt_meta_label"]
    clean_ws=True,
    key_col="review_key_cleanws",
)

print("Detected label columns:", detected_labels)
print("\nDuplicate rate (by cleaned key):",
      (d["review_key_cleanws"].duplicated().mean()))

print("\nSummary over duplicate groups (n>=2):")
display(dup_groups.describe(include="all"))

# 2) Text-only ceilings for each detected label column
print("\n=== Text-only ceiling (upper bound) by label source ===")
for col in detected_labels:
    try:
        ce = text_only_ceiling(d, "review_key_cleanws", col)
        print(f"{col}: {ce:.4f}")
    except Exception as e:
        print(f"{col}: [skip] {e}")

# 3) Pairwise majority agreement across label sources (per unique text)
print("\n=== Majority agreement per unique text (pairwise) ===")
for i in range(len(detected_labels)):
    for j in range(i+1, len(detected_labels)):
        a, b = detected_labels[i], detected_labels[j]
        agree, n_texts = majority_agreement(d, "review_key_cleanws", a, b)
        print(f"{a} vs {b}: agreement={agree:.4f} over {n_texts} unique texts")

# 4) OPTIONAL: metadata variation within duplicate text (only if these cols exist)
meta_cols = {"drug_id", "cond_id", "useful_z"}
if meta_cols.issubset(set(d.columns)):
    mv = dup_metadata_variation(d, "review_key_cleanws", "drug_id", "cond_id", "useful_z")
    print("\n=== Duplicate texts: how often metadata differs? ===")
    print("Share of duplicate groups where drug or condition varies:",
          float(mv["drug_or_cond_varies"].mean()))
else:
    print("\n[Info] Skipping metadata-variation check: need columns drug_id, cond_id, useful_z.")

Detected label columns: ['rating', 'sentiment_5', 'ai_rating_10', 'ai_sentiment_5', 'ai_prob_very_negative', 'ai_prob_very_positive', 'review_key_cleanws']

Duplicate rate (by cleaned key): 0.3833420274734829

Summary over duplicate groups (n>=2):


,review_key_cleanws,n,rating_nunique,rating_majshare,rating_entropy,sentiment_5_nunique,sentiment_5_majshare,sentiment_5_entropy,ai_rating_10_nunique,ai_rating_10_majshare,...,ai_sentiment_5_entropy,ai_prob_very_negative_nunique,ai_prob_very_negative_majshare,ai_prob_very_negative_entropy,ai_prob_very_positive_nunique,ai_prob_very_positive_majshare,ai_prob_very_positive_entropy,review_key_cleanws_nunique,review_key_cleanws_majshare,review_key_cleanws_entropy
count,10984,10984.000000,10984.000000,10984.000000,1.098400e+04,10984.000000,10984.000000,1.098400e+04,10984.000000,10984.000000,...,1.098400e+04,10984.000000,10984.000000,1.098400e+04,10984.000000,10984.000000,1.098400e+04,10984.0,10984.0,1.098400e+04
unique,10984,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,"""works very well.""",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,2.003551,1.001001,0.999686,5.519155e-04,1.000455,0.999868,2.526629e-04,1.388838,0.805887,...,1.369718e-01,1.319009,0.840579,2.210796e-01,1.640295,0.680586,4.432625e-01,1.0,1.0,-1.000089e-12
std,NaN,0.102265,0.041581,0.011802,2.090934e-02,0.025242,0.007306,1.375501e-02,0.488628,0.243621,...,2.759895e-01,0.466114,0.232977,3.230335e-01,0.483715,0.240237,3.336122e-01,0.0,0.0,7.088711e-26
min,NaN,2.000000,1.000000,0.500000,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.000000,0.444444,...,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.000000,0.250000,-1.000089e-12,1.0,1.0,-1.000089e-12
25%,NaN,2.000000,1.000000,1.000000,-1.000089e-12,1.000000,1.000000,-1.000089e-12,1.000000,0.500000,...,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.000000,0.500000,-1.000089e-12,1.0,1.0,-1.000089e-12
50%,NaN,2.000000,1.000000,1.000000,-1.000089e-12,1.000000,1.000000,-1.000089e-12,1.000000,1.000000,...,-1.000089e-12,1.000000,1.000000,-1.000089e-12,2.000000,0.500000,6.931472e-01,1.0,1.0,-1.000089e-12
75%,NaN,2.000000,1.000000,1.000000,-1.000089e-12,1.000000,1.000000,-1.000089e-12,2.000000,1.000000,...,-1.000089e-12,2.000000,1.000000,6.931472e-01,2.000000,1.000000,6.931472e-01,1.0,1.0,-1.000089e-12



=== Text-only ceiling (upper bound) by label source ===
rating: 0.9995
sentiment_5: 0.9997
ai_rating_10: 0.8513
ai_sentiment_5: 0.9243
ai_prob_very_negative: 0.8781
ai_prob_very_positive: 0.7553
review_key_cleanws: 1.0000

=== Majority agreement per unique text (pairwise) ===
rating vs sentiment_5: agreement=0.0000 over 17732 unique texts
rating vs ai_rating_10: agreement=0.2952 over 17732 unique texts
rating vs ai_sentiment_5: agreement=0.0000 over 17732 unique texts
rating vs ai_prob_very_negative: agreement=0.0000 over 17732 unique texts
rating vs ai_prob_very_positive: agreement=0.0000 over 17732 unique texts
rating vs review_key_cleanws: agreement=0.0000 over 17732 unique texts
sentiment_5 vs ai_rating_10: agreement=0.0000 over 17732 unique texts
sentiment_5 vs ai_sentiment_5: agreement=0.0000 over 17732 unique texts
sentiment_5 vs ai_prob_very_negative: agreement=0.0000 over 17732 unique texts
sentiment_5 vs ai_prob_very_positive: agreement=0.0000 over 17732 unique texts
sentime